In [ ]:
!pip install torch torchvision torchaudio
!pip install datasets transformers
!pip install g2p_en praatio librosa

In [1]:
import os
from os.path import exists, join, expanduser

os.chdir(expanduser("~"))
charsiu_dir = 'charsiu'
if exists(charsiu_dir):
  !rm -rf /root/charsiu
if not exists(charsiu_dir):
  ! git clone -b development https://github.com/lingjzhu/$charsiu_dir
  ! cd charsiu && git checkout && cd -
  
os.chdir(charsiu_dir)    

'rm' �����ڲ����ⲿ���Ҳ���ǿ����еĳ���
���������ļ���


In [2]:
import sys
import torch
from itertools import groupby
from datasets import load_dataset
import matplotlib.pyplot as plt

sys.path.insert(0,'src')
from Charsiu import charsiu_chain_attention_aligner, charsiu_chain_forced_aligner, charsiu_predictive_aligner

In [3]:
timit = load_dataset('timit_asr', data_dir="D:/ASR/charsiu/timit/data")

In [4]:
# load data
sample = timit['train'][0]
text = sample['text']
audio_path = sample['file']
print('Text transcription:%s'%(text))
print('Audio path: %s'%audio_path)

Text transcription:She had your dark suit in greasy wash water all year.
Audio path: D:\ASR\charsiu\timit\data\train\DR1\FCJF0\SA1.wav


Phone recognizer + Neural Forced Alignment

In [10]:
# load model
charsiu = charsiu_chain_attention_aligner(aligner='charsiu/en_w2v2_fs_10ms',recognizer='charsiu/en_w2v2_ctc_libris_and_cv')

d:\anaconda3\envs\charsiu\lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
d:\anaconda3\envs\charsiu\lib\site-packages\transformers\configuration_utils.py:369: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(
d:\anaconda3\envs\charsiu\lib\site-packages\torch\nn\utils\weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm

In [11]:
alignment = charsiu.align(audio=audio_path)

d:\ASR\charsiu\src\Charsiu.py:372: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  audio = torch.tensor(audio).float().unsqueeze(0).to(self.device)


In [12]:
print(alignment)
print('\n Ground Truth \n')
print([(s/16000,e/16000,p) for s,e,p in zip(sample['phonetic_detail']['start'],sample['phonetic_detail']['stop'],sample['phonetic_detail']['utterance'])])

[(0.0, 0.17, '[SIL]'), (0.17, 0.19, '[UNK]'), (0.19, 0.22, 'S'), (0.22, 0.23, '[UNK]'), (0.23, 0.29, 'S'), (0.29, 0.3, '[UNK]'), (0.3, 0.32, 'S'), (0.32, 0.33, '[UNK]'), (0.33, 0.34, 'Y'), (0.34, 0.38, 'S'), (0.38, 0.39, '[UNK]'), (0.39, 0.4, 'S'), (0.4, 0.44, 'L'), (0.44, 0.56, '[UNK]'), (0.56, 0.63, 'S'), (0.63, 0.93, '[UNK]'), (0.93, 1.01, 'K'), (1.01, 1.1, 'S'), (1.1, 1.23, '[UNK]'), (1.23, 1.27, 'L'), (1.27, 1.33, '[UNK]'), (1.33, 1.38, 'N'), (1.38, 1.42, 'D'), (1.42, 1.47, 'G'), (1.47, 1.52, 'R'), (1.52, 1.57, '[UNK]'), (1.57, 1.61, 'Y'), (1.61, 1.69, 'S'), (1.69, 1.71, '[UNK]'), (1.71, 1.79, 'Y'), (1.79, 1.87, 'W'), (1.87, 2.0, '[UNK]'), (2.0, 2.07, 'S'), (2.07, 2.1, '[UNK]'), (2.1, 2.18, 'W'), (2.18, 2.25, '[UNK]'), (2.25, 2.31, 'R'), (2.31, 2.46, '[UNK]'), (2.46, 2.52, 'L'), (2.52, 2.65, 'Y'), (2.65, 2.66, '[UNK]'), (2.66, 2.67, 'Y'), (2.67, 2.68, 'L'), (2.68, 2.8, '[UNK]'), (2.8, 2.9, '[SIL]')]

 Ground Truth 

[(0.0, 0.190625, 'h#'), (0.190625, 0.2849375, 'sh'), (0.2849375, 

In [13]:
charsiu.serve(audio=audio_path, save_to='sample.TextGrid')

Alignment output has been saved to sample.TextGrid


In [14]:
# load model
charsiu = charsiu_chain_forced_aligner(aligner='charsiu/en_w2v2_fc_10ms',recognizer='charsiu/en_w2v2_ctc_libris_and_cv')

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [15]:
alignment = charsiu.align(audio=audio_path)

d:\ASR\charsiu\src\Charsiu.py:457: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  audio = torch.tensor(audio).float().unsqueeze(0).to(self.device)


In [16]:
print(alignment)
print('\n Ground Truth \n')
print([(s/16000,e/16000,p) for s,e,p in zip(sample['phonetic_detail']['start'],sample['phonetic_detail']['stop'],sample['phonetic_detail']['utterance'])])

[(0.0, 0.23, '[SIL]'), (0.23, 0.27, '[UNK]'), (0.27, 0.28, 'S'), (0.28, 0.29, '[UNK]'), (0.29, 0.3, 'L'), (0.3, 0.31, '[UNK]'), (0.31, 0.32, 'S'), (0.32, 0.42, '[UNK]'), (0.42, 0.43, 'Y'), (0.43, 0.51, '[UNK]'), (0.51, 0.65, 'D'), (0.65, 0.7, '[UNK]'), (0.7, 0.84, 'D'), (0.84, 0.9, '[UNK]'), (0.9, 1.01, 'K'), (1.01, 1.14, 'S'), (1.14, 1.23, '[UNK]'), (1.23, 1.24, 'L'), (1.24, 1.26, '[UNK]'), (1.26, 1.37, 'N'), (1.37, 1.4, 'D'), (1.4, 1.44, 'G'), (1.44, 1.53, 'R'), (1.53, 1.57, '[UNK]'), (1.57, 1.58, 'Y'), (1.58, 1.69, 'S'), (1.69, 1.72, '[UNK]'), (1.72, 1.73, 'Y'), (1.73, 1.89, 'W'), (1.89, 1.97, '[UNK]'), (1.97, 1.98, 'S'), (1.98, 2.05, '[UNK]'), (2.05, 2.15, 'W'), (2.15, 2.17, '[UNK]'), (2.17, 2.31, 'R'), (2.31, 2.33, '[UNK]'), (2.33, 2.51, 'L'), (2.51, 2.67, 'Y'), (2.67, 2.69, '[UNK]'), (2.69, 2.81, 'R'), (2.81, 2.82, '[UNK]'), (2.82, 2.83, 'S'), (2.83, 2.84, '[UNK]'), (2.84, 2.85, 'L'), (2.85, 2.86, '[UNK]'), (2.86, 2.9, '[SIL]')]

 Ground Truth 

[(0.0, 0.190625, 'h#'), (0.190625,

In [17]:
charsiu.serve(audio=audio_path, save_to='sample.TextGrid')

Alignment output has been saved to sample.TextGrid


Direct inference with frame classification model

In [18]:
charsiu = charsiu_predictive_aligner(aligner='charsiu/en_w2v2_fc_10ms')

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [19]:
alignment = charsiu.align(audio=audio_path)

In [20]:
print(alignment)
print('\n Ground Truth \n')
print([(s/16000,e/16000,p) for s,e,p in zip(sample['phonetic_detail']['start'],sample['phonetic_detail']['stop'],sample['phonetic_detail']['utterance'])])

[(0.0, 0.16, '[SIL]'), (0.16, 0.28, 'SH'), (0.28, 0.36, 'IY'), (0.36, 0.44, 'HH'), (0.44, 0.54, 'AE'), (0.54, 0.62, 'D'), (0.62, 0.66, 'Y'), (0.66, 0.68, 'UH'), (0.68, 0.7, 'AH'), (0.7, 0.71, 'R'), (0.71, 0.82, 'D'), (0.82, 0.92, 'AH'), (0.92, 1.01, 'K'), (1.01, 1.13, 'S'), (1.13, 1.27, 'UW'), (1.27, 1.29, 'AH'), (1.29, 1.31, 'AE'), (1.31, 1.32, 'AH'), (1.32, 1.37, 'N'), (1.37, 1.4, 'D'), (1.4, 1.44, 'G'), (1.44, 1.51, 'R'), (1.51, 1.54, 'IH'), (1.54, 1.59, 'IY'), (1.59, 1.69, 'S'), (1.69, 1.74, 'IY'), (1.74, 1.84, 'W'), (1.84, 1.89, 'AO'), (1.89, 1.95, 'AA'), (1.95, 2.07, 'SH'), (2.07, 2.12, 'W'), (2.12, 2.21, 'AO'), (2.21, 2.23, 'R'), (2.23, 2.25, 'T'), (2.25, 2.34, 'ER'), (2.34, 2.43, 'AO'), (2.43, 2.51, 'L'), (2.51, 2.62, 'Y'), (2.62, 2.71, 'IH'), (2.71, 2.83, 'R'), (2.83, 2.9, '[SIL]')]

 Ground Truth 

[(0.0, 0.190625, 'h#'), (0.190625, 0.2849375, 'sh'), (0.2849375, 0.3576875, 'ix'), (0.3576875, 0.415125, 'hv'), (0.415125, 0.54825, 'eh'), (0.54825, 0.574375, 'dcl'), (0.574375, 0.

In [21]:
charsiu.serve(audio=audio_path, save_to='sample.TextGrid')

Alignment output has been saved to sample.TextGrid


In [4]:
import sys
from datasets import load_dataset
import matplotlib.pyplot as plt
sys.path.append('src/')
#sys.path.insert(0,'src')
from Charsiu import charsiu_forced_aligner, charsiu_attention_aligner
charsiu = charsiu_forced_aligner(aligner='charsiu/zh_w2v2_tiny_fc_10ms',lang='zh')
#charsiu_predictive_aligner(aligner='charsiu/zh_xlsr_fc_10ms', lang='zh')
charsiu.serve(audio='C:/Users/wangz/Desktop/wav/2_Annotation-Mandarin_16k.wav', text='除了质量问题，您说的那几家餐厅在服务水平上也是参差不齐的。',
              save_to='C:/Users/wangz/Desktop/wav/2_Annotation-Mandarin_16k.TextGrid')

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Alignment output has been saved to C:/Users/wangz/Desktop/wav/2_Annotation-Mandarin_16k.TextGrid


In [1]:
import sys
from datasets import load_dataset
import matplotlib.pyplot as plt
sys.path.append('src/')
#sys.path.insert(0,'src')
from Charsiu import charsiu_forced_aligner, charsiu_attention_aligner
charsiu = charsiu_forced_aligner(aligner='charsiu/zh_w2v2_tiny_fc_10ms', lang='zh')
#charsiu_predictive_aligner(aligner='charsiu/zh_xlsr_fc_10ms', lang='zh')
charsiu.serve(audio="C:/Users/wangz/Desktop/mfa_input/2.11_16k.mp3", text='我成长的过程中经历了各种各样的挑战，但这些经历让我不断改变，成为了更好的自己。以前我是个叛逆的孩子，不喜欢听别人的话，常常做一些让人头疼的事，谁劝我都没用。父母因此感到非常崩溃，他们觉得已经无法和我沟通，看着父母整天忧心忡忡，我心里也很矛盾。是继续这样下去，还是尝试去改变。这种挣扎持续了几个月，我甚至陷入了抑郁，那时候我几乎想要放弃一切。可是我父母始终没有放弃我，他们一直鼓励我，因为他们最了解我，也相信我能变得更好。慢慢的我开始有了目标，开始向往一个更美好的未来。在改变的过程中，我和父母一步步努力，有时我也会厌倦，抗拒他们的劝告，但我的毅力不允许我放弃。直到现在，我觉得自己已经成熟了许多，也变了更有责任感和纪律性，这就是我克服困难，改变自我，成长为更好自己的故事。',
              save_to='C:/Users/wangz/Desktop/mfa_input/2.11_16k.TextGrid')


d:\anaconda3\envs\charsiu\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\anaconda3\envs\charsiu\lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
d:\anaconda3\envs\charsiu\lib\site-packages\torch\nn\utils\weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Alignment output has been saved to C:/Users/wangz/Desktop/mfa_input/2.11_16k.TextGrid


In [8]:
# import essential libraries
import os
import sys

# change this path to where you saved the charsiu package
charsiu_dir = 'D:/ASR/charsiu'
os.chdir(charsiu_dir)

sys.path.append('%s/src/' % charsiu_dir)

# import selected model from Charsiu
from Charsiu import charsiu_predictive_aligner,charsiu_forced_aligner

# initialize model
charsiu = charsiu_predictive_aligner(aligner='charsiu/zh_xlsr_fc_10ms',lang='zh')

# load data
audio_file = "C:/Users/wangz/Desktop/wav/2_Annotation-Mandarin_16k.wav"
txt_file = 'C:/Users/wangz/Desktop/wav/2_Annotation-Mandarin_16k.txt'
textgrid_file = 'C:/Users/wangz/Desktop/wav/2_Annotation-Mandarin_16k.TextGrid'

# import selected model from Charsiu
from Charsiu import charsiu_forced_aligner

# initialize model
charsiu = charsiu_forced_aligner(aligner='charsiu/zh_xlsr_fc_10ms',lang='zh')

# read in text file
with open(txt_file) as f:
    text = f.read()

# perform forced alignment
alignment = charsiu.align(audio=audio_file,text=text)

# perform forced alignment and save the output as a textgrid file
charsiu.serve(audio=audio_file, text=text,
              save_to=textgrid_file)


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Alignment output has been saved to C:/Users/wangz/Desktop/wav/2_Annotation-Mandarin_16k.TextGrid


In [5]:
import os
import sys
sys.path.append('src/')

from Charsiu import charsiu_forced_aligner

charsiu = charsiu_forced_aligner(
    aligner='charsiu/zh_w2v2_tiny_fc_10ms',
    lang='zh'
)

data_dir = "C:/Users/wangz/Desktop/test"

for fname in os.listdir(data_dir):
    if not fname.lower().endswith(".wav"):
        continue

    wav_path = os.path.join(data_dir, fname)
    base = os.path.splitext(fname)[0]

    txt_path = os.path.join(data_dir, base + ".txt")
    tg_path  = os.path.join(data_dir, base + ".TextGrid")

    # ① 没有文本 → 跳过
    if not os.path.exists(txt_path):
        print(f"[跳过] 没有 txt: {base}")
        continue

    # ② 已经有 TextGrid → 跳过
    if os.path.exists(tg_path):
        print(f"[已存在] TextGrid 已有: {base}")
        continue

    # ③ 读取文本
    with open(txt_path, "r", encoding="utf-8") as f:
        text = f.read().strip()

    if not text:
        print(f"[跳过] 空文本: {base}")
        continue

    print(f"[对齐] {base}")

    try:
        charsiu.serve(
            audio=wav_path,
            text=text,
            save_to=tg_path
        )
    except Exception as e:
        print(f"[失败] {base}: {e}")


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


[跳过] 没有 txt: 2_Annotation-Mandarin
[对齐] 2_Annotation-Mandarin_16k
Alignment output has been saved to C:/Users/wangz/Desktop/test\2_Annotation-Mandarin_16k.TextGrid
[跳过] 没有 txt: 自-34-M-张昊.WAV_mono
[已存在] TextGrid 已有: 自-34-M-张昊.WAV_mono_16k
[跳过] 没有 txt: 自-34-M-赵豪.WAV_mono
[已存在] TextGrid 已有: 自-34-M-赵豪.WAV_mono_16k


In [2]:
import sys
from datasets import load_dataset
import matplotlib.pyplot as plt
sys.path.append('src/')
#sys.path.insert(0,'src')
from Charsiu import charsiu_forced_aligner, charsiu_attention_aligner
charsiu = charsiu_forced_aligner(aligner='charsiu/en_w2v2_fc_10ms')
#charsiu_predictive_aligner(aligner='charsiu/zh_xlsr_fc_10ms', lang='zh')
charsiu.serve(audio='C:/Users/wangz/Desktop/mfa_input/NAn1FwLsDC.wav', text='May launched herself into a journey of learning to reveal the wonders of ours.',
              save_to='C:/Users/wangz/Desktop/NAn1FwLsDC.TextGrid')

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Alignment output has been saved to C:/Users/wangz/Desktop/NAn1FwLsDC.TextGrid
